# Remaining Addition Sweeps on Dual T4 (Baselines and Alternative Architectures)

Finishes the random multi-carry sweeps that were too slow on CPU: 30 problems per length at 4, 8, 12, 14, 16, 18, 20, 25, 30, 35 digits,
greedy decoding, exact match of the final answer. Plain RoPE `MathGPT` models decode with a KV cache (verified token-identical to
uncached decoding); the other architectures decode uncached on GPU.

**Setup.** Add the dataset `cbt_eval_bundle` as an input (folders `code/`, `checkpoints/`, `results/`). The notebook assembles one repo root in
`/kaggle/working/repo`, resumes from the sweep JSONs in the bundle (finished lengths are skipped), and writes updated JSONs to
`/kaggle/working/repo/results/baseline_rerun/`. Download those three files and copy them over `results/baseline_rerun/` locally.

GPU 0 runs sweep A (place-value, sample-efficiency models) and sweep B (BPE philosophy); GPU 1 runs sweep C (five architectures).

In [ ]:
# Cell 1: hardware check
import torch, subprocess
print(torch.__version__, "cuda:", torch.cuda.is_available(), "gpus:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

In [ ]:
# Cell 2: assemble the repo root from the input dataset
import glob, os, shutil
src_code = glob.glob("/kaggle/input/**/code/src", recursive=True)
assert src_code, "dataset cbt_eval_bundle not attached"
BUNDLE = os.path.dirname(os.path.dirname(src_code[0]))
REPO = "/kaggle/working/repo"
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(os.path.join(BUNDLE, "code"), REPO)
shutil.copytree(os.path.join(BUNDLE, "checkpoints"), os.path.join(REPO, "checkpoints"))
shutil.copytree(os.path.join(BUNDLE, "results"), os.path.join(REPO, "results"))
print(BUNDLE, "->", REPO)
print(sorted(glob.glob(REPO + "/checkpoints/**/*.pt", recursive=True)))

In [ ]:
# Cell 3: smoke test (cache equals uncached decoding on the BPE model)
import subprocess
print(subprocess.run(["python", "-c", '''
import sys, random, torch
sys.path.insert(0, ".")
from benchmarks.evaluate_checkpoints import load_model_from_checkpoint, batched_greedy
from experiments.run_equalized_15d_1024_and_large_sweep import generate_random_problems_for_length
dev = torch.device("cuda:0")
m, t, c, _ = load_model_from_checkpoint("checkpoints/baselines_18m/bpe50k_1d_rope_wordcot_philo50_12d.pt", dev)
ps = [p + "<THINK>" for p, _ in generate_random_problems_for_length(8, 12, random.Random(1))]
g = {}
for p in ps: g.setdefault(len(t.encode(p, add_bos=True)), []).append(p)
pr = max(g.values(), key=len)[:4]
a = batched_greedy(m, t, pr, 60, dev, use_cache=False)
b = batched_greedy(m, t, pr, 60, dev, use_cache=True)
print("cache identical to uncached:", a == b)
assert a == b
'''], cwd=REPO, capture_output=True, text=True))

In [ ]:
# Cell 4: launch the sweeps in parallel (GPU 0: A then B, GPU 1: C); each resumes from the bundle JSONs
import subprocess, os
CK = "checkpoints/"
J = "results/baseline_rerun/"
cmd_ab = (
  f"python benchmarks/evaluate_checkpoints.py --n-per-length 30 --resume --out-json {J}sweep_a_char_2d.json --sweep "
  f"{CK}baselines_18m/char_1d_rope_placevalue_grpo_12d.pt {CK}sample_efficiency/hrm_2d_rope_k32_n1000.pt "
  f"{CK}sample_efficiency/hrm_2d_rope_k32_n100.pt {CK}sample_efficiency/hrm_2d_rope_k32_n10.pt > sweep_a.log 2>&1; "
  f"python benchmarks/evaluate_checkpoints.py --n-per-length 30 --resume --out-json {J}sweep_b_bpe.json --sweep "
  f"{CK}baselines_18m/bpe50k_1d_rope_wordcot_philo50_12d.pt > sweep_b.log 2>&1"
)
cmd_c = (
  f"python benchmarks/evaluate_checkpoints.py --n-per-length 30 --resume --out-json {J}sweep_c_arch.json --sweep "
  f"{CK}architectures_char/looped2x6_1d_rope_12d.pt {CK}architectures_char/hierarchical_f4s3_1d_rope_12d.pt "
  f"{CK}architectures_char/hybrid_glru_attn_1d_rope_12d.pt {CK}architectures_char/hnrm_harmonic_12d.pt "
  f"{CK}architectures_char/alsm_looped4x6_abacus_k32_12d.pt > sweep_c.log 2>&1"
)
procs = [
  subprocess.Popen(cmd_ab, shell=True, cwd=REPO, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"}),
  subprocess.Popen(cmd_c, shell=True, cwd=REPO, env={**os.environ, "CUDA_VISIBLE_DEVICES": "1"}),
]
import time, subprocess as sp
while any(p.poll() is None for p in procs):
    time.sleep(300)
    for log in ("sweep_a.log", "sweep_b.log", "sweep_c.log"):
        path = os.path.join(REPO, log)
        if os.path.exists(path):
            tail = [l for l in open(path).read().splitlines() if "[INFO] Sweep" in l or "seen" in l][-2:]
            print(log, "|", " || ".join(t.strip() for t in tail), flush=True)
print("exit codes:", [p.returncode for p in procs])

In [ ]:
# Cell 5: summary table (correct / n per length) and files to download
import json, glob
for f in sorted(glob.glob(REPO + "/results/baseline_rerun/sweep_*.json")):
    d = json.load(open(f))
    print("==", os.path.basename(f))
    for r in d["results"]:
        row = "  ".join(f"{k}d:{v['correct']}/{v['n']}" for k, v in r["per_length"].items())
        print(f"{os.path.basename(r['checkpoint'])[:44]:44s} {row}")
for f in sorted(glob.glob(REPO + "/sweep_*.log")):
    print(f, os.path.getsize(f), "bytes")

In [ ]:
# Cell 6: copy the deliverables to the notebook output root for download
import shutil
for f in glob.glob(REPO + "/results/baseline_rerun/sweep_*.json") + glob.glob(REPO + "/sweep_*.log"):
    shutil.copy(f, "/kaggle/working/")
print(sorted(os.listdir("/kaggle/working")))